# this notebook is for experimenting and model training

In [1]:
%load_ext autoreload
%autoreload 2
import math
import torch
import json
import torch.optim as optim
from pathlib import Path
from pet_breed_classifier.data import get_dataloaders
from pet_breed_classifier.model import get_model, get_criterion, get_optimizer, train_one_epoch, evaluate

## Sanity check for model
we have 37 classes and in each batch 32 images so output shape should be [32, 37]

before training model should be guessing evenly on each class, so loss for untraind model should be around ln(37) ~ 3.61

In [2]:
root = Path("../data").resolve()
train_loader, val_loader, test_loader, classes = get_dataloaders(root)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

model = get_model(len(classes), device, freeze=True)
criterion = get_criterion()

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

model.eval()
with torch.no_grad():
    logits = model(images)
    loss = criterion(logits, labels)

print(logits.shape)
print(loss.item())

torch.Size([32, 37])
3.764716148376465


## train and validate model(freeze backbone)

epoch 0/14 | train 2.038/ 0.536 | val 0.994/ 0.795 

epoch 1/14 | train 0.809/ 0.844 | val 0.626/ 0.863 

epoch 2/14 | train 0.544/ 0.892 | val 0.497/ 0.871 

epoch 3/14 | train 0.436/ 0.906 | val 0.450/ 0.852 

epoch 4/14 | train 0.369/ 0.917 | val 0.399/ 0.897 

epoch 5/14 | train 0.316/ 0.927 | val 0.363/ 0.890 

epoch 6/14 | train 0.278/ 0.933 | val 0.351/ 0.897 

epoch 7/14 | train 0.254/ 0.939 | val 0.339/ 0.891 

epoch 8/14 | train 0.242/ 0.946 | val 0.313/ 0.902 

epoch 9/14 | train 0.231/ 0.941 | val 0.335/ 0.891 

epoch 10/14 | train 0.214/ 0.945 | val 0.302/ 0.898 

epoch 11/14 | train 0.201/ 0.951 | val 0.321/ 0.891 

epoch 12/14 | train 0.190/ 0.954 | val 0.306/ 0.899 

epoch 13/14 | train 0.175/ 0.954 | val 0.309/ 0.889 

epoch 14/14 | train 0.168/ 0.959 | val 0.319/ 0.882 

training loss keeps deacreasing for whole process so learning rate is stable,training accuracy reaches almost 85% after 2 epoches that is because we are using pretrained model that just needs to learn differentiation between our classes, validation accuracy plateus after 4th epoch, and  that seems to be limit for our linear head on frozen features, overfiting appears in later epochs

In [3]:
epochs = 15

model = get_model(len(classes), device, freeze=True)
parameters = [param for param in model.parameters() if param.requires_grad]
optimizer = get_optimizer(parameters, lr=1e-3)

model_parameters_path = Path("../parameters/model1/parameters.pt").resolve()
top_acc = 0.0

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for i in range(epochs):
    train_loss, train_acc = train_one_epoch(model, criterion, optimizer, train_loader, device)
    val_loss, val_acc = evaluate(model, criterion, val_loader, device)

    for k, v in zip(history, (train_loss, train_acc, val_loss, val_acc)):
        history[k].append(v)

    print(f"epoch {i}/{epochs-1} | train {train_loss:.3f}/ {train_acc:.3f} | val {val_loss:.3f}/ {val_acc:.3f} ")
    
    if val_acc > top_acc:
        top_acc = val_acc
        torch.save(model.state_dict(), model_parameters_path)

epoch 0/14 | train 2.095/ 0.530 | val 1.045/ 0.781 
epoch 1/14 | train 0.810/ 0.851 | val 0.614/ 0.870 
epoch 2/14 | train 0.549/ 0.893 | val 0.513/ 0.876 
epoch 3/14 | train 0.432/ 0.905 | val 0.440/ 0.872 
epoch 4/14 | train 0.372/ 0.916 | val 0.401/ 0.880 
epoch 5/14 | train 0.321/ 0.925 | val 0.389/ 0.883 
epoch 6/14 | train 0.286/ 0.935 | val 0.358/ 0.893 
epoch 7/14 | train 0.255/ 0.944 | val 0.346/ 0.891 
epoch 8/14 | train 0.245/ 0.941 | val 0.329/ 0.898 
epoch 9/14 | train 0.228/ 0.945 | val 0.318/ 0.904 
epoch 10/14 | train 0.202/ 0.950 | val 0.332/ 0.887 
epoch 11/14 | train 0.206/ 0.951 | val 0.316/ 0.895 
epoch 12/14 | train 0.177/ 0.957 | val 0.302/ 0.893 
epoch 13/14 | train 0.177/ 0.959 | val 0.297/ 0.897 
epoch 14/14 | train 0.165/ 0.958 | val 0.276/ 0.909 


In [4]:
json.dump(history, open("../reports/frozen.json", "w"))

## train and evaluate model(fine-tune)
#### first experiment:
epochs=10
"params": backbone_params, "lr":1e-04}, {"params": model.fc.parameters(), "lr":1e-03}

epoch 0/9 | train 0.304/ 0.899 | val 0.348/ 0.887 

epoch 1/9 | train 0.124/ 0.966 | val 0.299/ 0.901 

epoch 2/9 | train 0.076/ 0.984 | val 0.299/ 0.902 

epoch 3/9 | train 0.057/ 0.988 | val 0.296/ 0.913 

epoch 4/9 | train 0.044/ 0.990 | val 0.330/ 0.901 

epoch 5/9 | train 0.034/ 0.993 | val 0.398/ 0.890 

epoch 6/9 | train 0.057/ 0.983 | val 0.410/ 0.870 

epoch 7/9 | train 0.034/ 0.991 | val 0.406/ 0.878 

epoch 8/9 | train 0.032/ 0.992 | val 0.347/ 0.901 

epoch 9/9 | train 0.030/ 0.992 | val 0.428/ 0.879 

in this case validation loss starts bounciing after 3rd epoch, so after that, model starts overfiting, also we can assume that learning rate for backbone params are high -  because as we see training loss has increasing between 5th and 6th epochs, so for next experiment we can try to train for less epochs or with less learning rate for backbone params 

In [5]:
model_parameters_path_fn = Path("../parameters/model2/parameters.pt").resolve()

model = get_model(len(classes), device, freeze=False)
model.load_state_dict(torch.load(model_parameters_path, map_location=device))

backbone_params = [par for name, par in model.named_parameters() if not name.startswith("fc.")]
optimizer = optim.Adam([{"params": backbone_params, "lr":1e-04}, {"params": model.fc.parameters(), "lr":1e-03}])

top_acc = 0.0
epochs = 10

history_ft = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for i in range(epochs):
    train_loss, train_acc = train_one_epoch(model, criterion, optimizer, train_loader, device)
    val_loss, val_acc = evaluate(model, criterion, val_loader, device)

    for k, v in zip(history_ft, (train_loss, train_acc, val_loss, val_acc)):
        history_ft[k].append(v)

    print(f"epoch {i}/{epochs-1} | train {train_loss:.3f}/ {train_acc:.3f} | val {val_loss:.3f}/ {val_acc:.3f} ")
    
    if val_acc > top_acc:
        top_acc = val_acc
        torch.save(model.state_dict(), model_parameters_path_fn)
        
with open("../reports/fine_tuned.json", "w") as f:    
    json.dump(history_ft, f)

epoch 0/9 | train 0.228/ 0.930 | val 0.403/ 0.870 
epoch 1/9 | train 0.139/ 0.954 | val 0.367/ 0.874 
epoch 2/9 | train 0.081/ 0.975 | val 0.356/ 0.895 
epoch 3/9 | train 0.049/ 0.986 | val 0.371/ 0.895 
epoch 4/9 | train 0.050/ 0.989 | val 0.311/ 0.901 
epoch 5/9 | train 0.033/ 0.992 | val 0.327/ 0.904 
epoch 6/9 | train 0.039/ 0.988 | val 0.370/ 0.905 
epoch 7/9 | train 0.034/ 0.991 | val 0.439/ 0.891 
epoch 8/9 | train 0.049/ 0.986 | val 0.457/ 0.890 
epoch 9/9 | train 0.049/ 0.985 | val 0.532/ 0.848 


#### second experiment:
epochs=10 "params": backbone_params, "lr":3e-05}, {"params": model.fc.parameters(), "lr":1e-03}

lowering backbone learning rate lowered validation loss and delayed overfiting, so our asumption about this was right, train accuracy still reaches 99.8, so model still memorises training examples

epoch 0/9 | train 0.247/ 0.932 | val 0.276/ 0.917 

epoch 1/9 | train 0.133/ 0.968 | val 0.311/ 0.893 

epoch 2/9 | train 0.086/ 0.984 | val 0.274/ 0.901 

epoch 3/9 | train 0.073/ 0.983 | val 0.258/ 0.906 

epoch 4/9 | train 0.054/ 0.989 | val 0.244/ 0.917 

epoch 5/9 | train 0.038/ 0.994 | val 0.255/ 0.918 

epoch 6/9 | train 0.031/ 0.997 | val 0.257/ 0.920 

epoch 7/9 | train 0.023/ 0.998 | val 0.255/ 0.916 

epoch 8/9 | train 0.022/ 0.998 | val 0.319/ 0.909 

epoch 9/9 | train 0.019/ 0.998 | val 0.280/ 0.908 

In [6]:
model_parameters_path_fn = Path("../parameters/model2/parameters2.pt").resolve()
frozen_path = Path("../parameters/model1/parameters.pt").resolve()

model = get_model(len(classes), device, freeze=False)
model.load_state_dict(torch.load(frozen_path, map_location=device))

backbone_params = [par for name, par in model.named_parameters() if not name.startswith("fc.")]
optimizer = optim.Adam([{"params": backbone_params, "lr":3e-05}, {"params": model.fc.parameters(), "lr":1e-03}])

top_acc = 0.0
epochs = 10

history_ft = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for i in range(epochs):
    train_loss, train_acc = train_one_epoch(model, criterion, optimizer, train_loader, device)
    val_loss, val_acc = evaluate(model, criterion, val_loader, device)

    for k, v in zip(history_ft, (train_loss, train_acc, val_loss, val_acc)):
        history_ft[k].append(v)

    print(f"epoch {i}/{epochs-1} | train {train_loss:.3f}/ {train_acc:.3f} | val {val_loss:.3f}/ {val_acc:.3f} ")
    
    if val_acc > top_acc:
        top_acc = val_acc
        torch.save(model.state_dict(), model_parameters_path_fn)
        
with open("../reports/history_ft_lr_3e-5.json", "a") as f:    
    json.dump(history_ft, f)

epoch 0/9 | train 0.178/ 0.949 | val 0.290/ 0.898 
epoch 1/9 | train 0.104/ 0.977 | val 0.288/ 0.901 
epoch 2/9 | train 0.078/ 0.983 | val 0.283/ 0.905 
epoch 3/9 | train 0.050/ 0.990 | val 0.284/ 0.905 
epoch 4/9 | train 0.044/ 0.991 | val 0.287/ 0.902 
epoch 5/9 | train 0.034/ 0.993 | val 0.281/ 0.910 
epoch 6/9 | train 0.028/ 0.995 | val 0.282/ 0.909 
epoch 7/9 | train 0.024/ 0.994 | val 0.302/ 0.906 
epoch 8/9 | train 0.021/ 0.997 | val 0.281/ 0.918 
epoch 9/9 | train 0.019/ 0.997 | val 0.283/ 0.917 


#### third experiment:
epochs=10 "params": backbone_params, "lr":1e-05}, {"params": model.fc.parameters(), "lr":1e-03}

epoch 0/9 | train 0.235/ 0.935 | val 0.313/ 0.899 

epoch 1/9 | train 0.169/ 0.955 | val 0.295/ 0.899 

epoch 2/9 | train 0.130/ 0.968 | val 0.268/ 0.904 

epoch 3/9 | train 0.116/ 0.972 | val 0.259/ 0.910 

epoch 4/9 | train 0.085/ 0.982 | val 0.256/ 0.906 

epoch 5/9 | train 0.071/ 0.985 | val 0.262/ 0.904 

epoch 6/9 | train 0.057/ 0.990 | val 0.257/ 0.908 

epoch 7/9 | train 0.054/ 0.990 | val 0.279/ 0.902 

epoch 8/9 | train 0.044/ 0.995 | val 0.255/ 0.912 

epoch 9/9 | train 0.041/ 0.993 | val 0.258/ 0.920 

In [7]:
model_parameters_path_fn = Path("../parameters/model2/parameters3.pt").resolve()
frozen_path = Path("../parameters/model1/parameters.pt").resolve()

model = get_model(len(classes), device, freeze=False)
model.load_state_dict(torch.load(frozen_path, map_location=device))

backbone_params = [par for name, par in model.named_parameters() if not name.startswith("fc.")]
optimizer = optim.Adam([{"params": backbone_params, "lr":1e-05}, {"params": model.fc.parameters(), "lr":1e-03}])

top_acc = 0.0
epochs = 10

history_ft = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for i in range(epochs):
    train_loss, train_acc = train_one_epoch(model, criterion, optimizer, train_loader, device)
    val_loss, val_acc = evaluate(model, criterion, val_loader, device)

    for k, v in zip(history_ft, (train_loss, train_acc, val_loss, val_acc)):
        history_ft[k].append(v)

    print(f"epoch {i}/{epochs-1} | train {train_loss:.3f}/ {train_acc:.3f} | val {val_loss:.3f}/ {val_acc:.3f} ")
    
    if val_acc > top_acc:
        top_acc = val_acc
        torch.save(model.state_dict(), model_parameters_path_fn)
        
with open("../reports/history_ft_lr_1e-5.json", "w") as f:    
    json.dump(history_ft, f)

epoch 0/9 | train 0.177/ 0.949 | val 0.289/ 0.908 
epoch 1/9 | train 0.122/ 0.971 | val 0.274/ 0.905 
epoch 2/9 | train 0.100/ 0.978 | val 0.274/ 0.910 
epoch 3/9 | train 0.080/ 0.983 | val 0.287/ 0.894 
epoch 4/9 | train 0.062/ 0.988 | val 0.263/ 0.898 
epoch 5/9 | train 0.062/ 0.989 | val 0.288/ 0.899 
epoch 6/9 | train 0.052/ 0.990 | val 0.269/ 0.906 
epoch 7/9 | train 0.044/ 0.993 | val 0.268/ 0.899 
epoch 8/9 | train 0.036/ 0.997 | val 0.271/ 0.912 
epoch 9/9 | train 0.039/ 0.992 | val 0.260/ 0.902 


from this infrmation i would choose model from second experiment it had best validation accuracy along with third one, but reached lowest loss

In [8]:
frozen = Path("../parameters/model1/parameters.pt").resolve()
ft_lr_1e_minus4 = Path("../parameters/model2/parameters.pt").resolve()
ft_lr_3e_minus5 = Path("../parameters/model2/parameters2.pt").resolve()
ft_lr_1e_minus5 = Path("../parameters/model2/parameters3.pt").resolve()

models_weights = [("frozen", frozen), ("ft_lr_1e_minus4", ft_lr_1e_minus4), ("ft_lr_3e_minus5", ft_lr_3e_minus5), ("ft_lr_1e_minus5", ft_lr_1e_minus5)]

results = {}
for name, path in models_weights:
    m = get_model(len(classes), device)
    m.load_state_dict(torch.load(path, map_location=device))
    results[name] = evaluate(m, criterion, test_loader, device)
    print(f"{name:10s} | test loss {results[name][0]:.3f} | test acc {results[name][1]:.3f}")

frozen     | test loss 0.368 | test acc 0.885
ft_lr_1e_minus4 | test loss 0.461 | test acc 0.875
ft_lr_3e_minus5 | test loss 0.365 | test acc 0.899
ft_lr_1e_minus5 | test loss 0.350 | test acc 0.897


selected model (backbone LR 3e-5, chosen by lowest validation loss) reaches 89.7% test accuracy vs 87.9% for frozen baseline. the 1e-5 model scored 89.8% this 0.1 point difference is well within the 0.5 point standard error, so the two are equivalent.

frozen     | test loss 0.391 | test acc 0.879
ft_lr_1e_minus4 | test loss 0.416 | test acc 0.881
ft_lr_3e_minus5 | test loss 0.351 | test acc 0.897
ft_lr_1e_minus5 | test loss 0.349 | test acc 0.898